# Buscando Diferencias entre Grupos Independientes

## ANOVA de 2 factores para grupos independientes

### Procedimiento
- Importar librerias
- Cargar la primera hoja de trabajo de excel en Pandas
- Limpiar los datos
- Dividir el conjunto de datos en categorias (variables cualitativas independientes)
- Prueba de Normalidad (SHAPIRO-WILK)
- Prueba de homocedasticidad (LEVENE)
- Prueba ANOVA 2 Factores
- (opcional) Para la categoria con 3 o mas grupos HSD Tukey

In [ ]:
#__ Cargar librerias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as sps
import statsmodels.api as stmdls
from statsmodels.formula.api import ols

In [ ]:
#___ Cargar la hoja de trabajo en un dataframe
anova_2_df = pd.read_excel('datos_analisis_estadistico.xlsx', sheet_name = 'ANOVA 2 FACTORES')

In [ ]:
#___ Limpiar el dataframe
anova_2_df.columns = ['ELEMENTO', 'SEXO', 'EJERCICIO', 'IMC']
anova_2_df = anova_2_df.drop('ELEMENTO', axis=1)
anova_2_df

In [ ]:
#___ Dividir los datos en categorias (variables cualitativas independientes)
anova_2_mn_df = anova_2_df[(anova_2_df['SEXO'] == 'MASCULINO') & (anova_2_df['EJERCICIO'] == 'NUNCA')]
anova_2_me_df = anova_2_df[(anova_2_df['SEXO'] == 'MASCULINO') & (anova_2_df['EJERCICIO'] == 'EVENTUAL')]
anova_2_ms_df = anova_2_df[(anova_2_df['SEXO'] == 'MASCULINO') & (anova_2_df['EJERCICIO'] == 'SIEMPRE')]
anova_2_fn_df = anova_2_df[(anova_2_df['SEXO'] == 'FEMENINO') & (anova_2_df['EJERCICIO'] == 'NUNCA')]
anova_2_fe_df = anova_2_df[(anova_2_df['SEXO'] == 'FEMENINO') & (anova_2_df['EJERCICIO'] == 'EVENTUAL')]
anova_2_fs_df = anova_2_df[(anova_2_df['SEXO'] == 'FEMENINO') & (anova_2_df['EJERCICIO'] == 'SIEMPRE')]

In [ ]:
#___ Prueba de normalidad ANOVA (SHAPIRO-WILK)
prueba_norm = sps.shapiro(anova_2_df['IMC'])
prueba_norm

In [ ]:
#___ Prueba de homocedasticidad (LEVENE)
prueba_homos = sps.levene(anova_2_mn_df['IMC'], anova_2_me_df['IMC'], anova_2_ms_df['IMC'], anova_2_fn_df['IMC'], anova_2_fe_df['IMC'], anova_2_fs_df['IMC'])
prueba_homos

In [ ]:
#___ Prueba ANOVA 2 Factores
modelo = ols(formula = 'IMC ~ C(SEXO) + C(EJERCICIO) + C(SEXO):C(EJERCICIO)', data = anova_2_df).fit()
prueba_anova_2 = stmdls.stats.anova_lm(modelo, type=2)
prueba_anova_2

In [ ]:
#___ Prueba Tukey
prueba_tukey = sps.tukey_hsd(anova_2_df[anova_2_df['EJERCICIO'] == 'NUNCA']['IMC'], anova_2_df[anova_2_df['EJERCICIO'] == 'EVENTUAL']['IMC'], anova_2_df[anova_2_df['EJERCICIO'] == 'SIEMPRE']['IMC'])
print(prueba_tukey)


In [ ]:
#___ Grafica de error promedio
##___ Prepararr el dataframe para los graficos
promedios = anova_2_df.groupby(['EJERCICIO','SEXO']).mean()
destd = anova_2_df.groupby(['EJERCICIO','SEXO']).std()
##___ unstack()
promedios_un = promedios['IMC'].unstack()
destd_un = destd['IMC'].unstack()
###___ graficar
promedios_un.plot(kind='bar', yerr=destd_un, capsize=5, rot=1, figsize=(3, 4))
plt.xlabel('TEMPORALIDAD')
plt.ylabel('IMC')
plt.title('ERROR PROMEDIO POR GENERO')
plt.ylim(25,35)